In [2]:
# ============================================================
# MULTI-STATION WATER TABLE FLUCTUATION RECHARGE ANALYSIS
# ============================================================
#
# WTF equation:
#
# Recharge = Sy × water-table rise
#
# Daily recharge:
# Recharge_mm_day = Sy × positive daily rise in metres × 1000
#
# Annual recharge:
# Recharge_mm_year = sum of daily recharge within each year
#
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, FileLink


# ============================================================
# 2. INPUT AND OUTPUT PATHS
# ============================================================

gwl_folder = Path(
    "/kaggle/input/datasets/kausar15027/gwl-data/"
    "gw_levels_1980_2022_min_30y"
)

station_file = Path(
    "/kaggle/input/datasets/kausar15027/station-name1/Station_ID.csv"
)

output_folder = Path(
    "/kaggle/working/WTF_Recharge_Results"
)

zip_file = Path(
    "/kaggle/working/WTF_Recharge_Complete_Results.zip"
)


# ============================================================
# 3. ANALYSIS SETTINGS
# ============================================================

start_date = "2010-01-01"
end_date = "2022-12-31"

# Use:
# "head"  = larger GWL value means water table rises
# "depth" = smaller GWL value means water table rises
gwl_type = "head"

# Input groundwater-level unit
# Options: "m", "cm", "mm"
gwl_unit = "m"

# Ignore groundwater rises smaller than this value
minimum_rise_m = 0.002

# Interpolate gaps up to this number of days
maximum_gap_days = 7

# Median smoothing window
# Set to 1 to disable smoothing
smoothing_days = 3

# Minimum annual data coverage required
minimum_annual_coverage = 80

# Plot quality
plot_dpi = 300


# ============================================================
# 4. CREATE OUTPUT FOLDERS
# ============================================================

if output_folder.exists():
    shutil.rmtree(output_folder)

daily_folder = output_folder / "01_Daily_Recharge"
monthly_folder = output_folder / "02_Monthly_Recharge"
annual_folder = output_folder / "03_Annual_Recharge"
plot_folder = output_folder / "04_Plots"
log_folder = output_folder / "05_Logs"

for folder in [
    daily_folder,
    monthly_folder,
    annual_folder,
    plot_folder,
    log_folder
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 5. FLEXIBLE CSV READER
# ============================================================

def read_csv_flexible(path):
    """
    Read CSV files using several possible encodings and separators.
    """

    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp1252",
        "latin1"
    ]

    separators = [
        ",",
        ";",
        "\t",
        None
    ]

    last_error = None

    for encoding in encodings:

        for separator in separators:

            try:

                dataframe = pd.read_csv(
                    path,
                    encoding=encoding,
                    sep=separator,
                    engine="python"
                )

                if dataframe.shape[1] >= 1:
                    return dataframe

            except Exception as error:
                last_error = error

    raise ValueError(
        f"Could not read file: {path}\n"
        f"Last error: {last_error}"
    )


# ============================================================
# 6. COLUMN CLEANING FUNCTIONS
# ============================================================

def clean_column_names(dataframe):
    """
    Remove hidden characters, spaces and line breaks.
    """

    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("\n", "", regex=False)
        .str.replace("\r", "", regex=False)
        .str.strip()
    )

    return dataframe


def normalized_name(name):
    """
    Normalize a column name for matching.
    """

    return (
        str(name)
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
        .strip()
    )


# ============================================================
# 7. DETECT STATION ID AND Sy COLUMNS
# ============================================================

def find_station_columns(dataframe):
    """
    Detect station ID and specific-yield columns.
    """

    lookup = {
        normalized_name(column): column
        for column in dataframe.columns
    }

    id_column = None
    sy_column = None

    id_names = [
        "gwid",
        "stationid",
        "wellid",
        "id"
    ]

    sy_names = [
        "sy",
        "specificyield"
    ]

    for name in id_names:

        if name in lookup:
            id_column = lookup[name]
            break

    for name in sy_names:

        if name in lookup:
            sy_column = lookup[name]
            break

    # Safe fallback
    if id_column is None and dataframe.shape[1] >= 1:
        id_column = dataframe.columns[0]

    if sy_column is None and dataframe.shape[1] >= 2:
        sy_column = dataframe.columns[1]

    if id_column is None or sy_column is None:

        raise ValueError(
            "Could not identify station ID and Sy columns.\n"
            f"Available columns: {dataframe.columns.tolist()}"
        )

    return id_column, sy_column


# ============================================================
# 8. DETECT DATE AND GWL COLUMNS
# ============================================================

def detect_gwl_columns(dataframe):
    """
    Detect date and groundwater-level columns.
    """

    dataframe = clean_column_names(dataframe)

    lookup = {
        normalized_name(column): column
        for column in dataframe.columns
    }

    date_candidates = [
        "date",
        "datetime",
        "timestamp",
        "time"
    ]

    gwl_candidates = [
        "gwl",
        "groundwaterlevel",
        "waterlevel",
        "head",
        "depth",
        "value"
    ]

    date_column = None
    gwl_column = None

    for name in date_candidates:

        if name in lookup:
            date_column = lookup[name]
            break

    for name in gwl_candidates:

        if name in lookup:
            gwl_column = lookup[name]
            break

    # Fallback for date column
    if date_column is None:

        best_rate = 0

        for column in dataframe.columns:

            converted = pd.to_datetime(
                dataframe[column],
                errors="coerce"
            )

            success_rate = converted.notna().mean()

            if success_rate > best_rate:

                best_rate = success_rate
                date_column = column

        if best_rate < 0.70:
            date_column = None

    # Fallback for GWL column
    if gwl_column is None:

        best_rate = 0

        for column in dataframe.columns:

            if column == date_column:
                continue

            converted = pd.to_numeric(
                dataframe[column],
                errors="coerce"
            )

            success_rate = converted.notna().mean()

            if success_rate > best_rate:

                best_rate = success_rate
                gwl_column = column

        if best_rate < 0.70:
            gwl_column = None

    return date_column, gwl_column


# ============================================================
# 9. OTHER HELPER FUNCTIONS
# ============================================================

def extract_station_id(filename):
    """
    Extract station ID from the beginning of a filename.

    Example:
    49471239_1980_2022.csv -> 49471239
    """

    match = re.match(
        r"^\s*(\d+)",
        filename
    )

    if match:
        return match.group(1)

    return None


def convert_to_metres(values, unit):
    """
    Convert groundwater-level values to metres.
    """

    unit = unit.lower().strip()

    if unit == "m":
        return values

    if unit == "cm":
        return values / 100

    if unit == "mm":
        return values / 1000

    raise ValueError(
        "gwl_unit must be 'm', 'cm', or 'mm'."
    )


# ============================================================
# 10. READ STATION ID AND SPECIFIC YIELD
# ============================================================

station_df = read_csv_flexible(
    station_file
)

station_df = clean_column_names(
    station_df
)

print("Detected columns in Station_ID.csv:")
print(station_df.columns.tolist())

display(
    station_df.head()
)

id_column, sy_column = find_station_columns(
    station_df
)

print(
    f"\nStation ID column used: {id_column}"
)

print(
    f"Specific-yield column used: {sy_column}"
)

station_lookup = pd.DataFrame()

station_lookup["station_id"] = (
    station_df[id_column]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\.0$",
        "",
        regex=True
    )
)

station_lookup["Sy"] = pd.to_numeric(
    station_df[sy_column],
    errors="coerce"
)

station_lookup = station_lookup.dropna(
    subset=[
        "station_id",
        "Sy"
    ]
)

station_lookup = station_lookup[
    station_lookup["station_id"] != ""
]

station_lookup = station_lookup.drop_duplicates(
    subset="station_id",
    keep="first"
)

sy_dictionary = dict(
    zip(
        station_lookup["station_id"],
        station_lookup["Sy"]
    )
)

print(
    f"\nValid station-Sy records: "
    f"{len(sy_dictionary)}"
)

display(
    station_lookup.head(10)
)


# ============================================================
# 11. PROCESS ALL GROUNDWATER FILES
# ============================================================

all_daily = []
all_monthly = []
all_annual = []

summary_records = []
processing_log = []

gwl_files = sorted(
    gwl_folder.glob("*.csv")
)

print(
    f"\nGroundwater files found: "
    f"{len(gwl_files)}"
)

for file_number, file_path in enumerate(
    gwl_files,
    start=1
):

    station_id = extract_station_id(
        file_path.name
    )

    print(
        f"\nProcessing {file_number}/{len(gwl_files)}: "
        f"{file_path.name}"
    )

    if station_id is None:

        processing_log.append({

            "file": file_path.name,

            "station_id": "",

            "status": "Skipped",

            "message":
                "Could not extract station ID from filename"
        })

        continue

    if station_id not in sy_dictionary:

        processing_log.append({

            "file": file_path.name,

            "station_id": station_id,

            "status": "Skipped",

            "message":
                "Station ID not found in station Sy table"
        })

        continue

    try:

        sy = float(
            sy_dictionary[station_id]
        )

        dataframe = read_csv_flexible(
            file_path
        )

        dataframe = clean_column_names(
            dataframe
        )

        date_column, gwl_column = detect_gwl_columns(
            dataframe
        )

        if date_column is None or gwl_column is None:

            raise ValueError(
                "Could not identify date or groundwater-level column.\n"
                f"Available columns: {dataframe.columns.tolist()}"
            )

        data = pd.DataFrame()

        data["date"] = pd.to_datetime(
            dataframe[date_column],
            errors="coerce"
        )

        data["gwl_original"] = pd.to_numeric(
            dataframe[gwl_column],
            errors="coerce"
        )

        data = data.dropna(
            subset=[
                "date",
                "gwl_original"
            ]
        )

        data = data.sort_values(
            "date"
        )

        data = data[
            (data["date"] >= start_date)
            &
            (data["date"] <= end_date)
        ].copy()

        if data.empty:

            raise ValueError(
                "No valid groundwater-level data "
                "within the selected period."
            )

        data["gwl_m"] = convert_to_metres(
            data["gwl_original"],
            gwl_unit
        )


        # ====================================================
        # DAILY GROUNDWATER LEVEL
        # ====================================================

        daily_gwl = (
            data
            .set_index("date")["gwl_m"]
            .resample("D")
            .mean()
        )

        daily_gwl = daily_gwl.interpolate(
            method="time",
            limit=maximum_gap_days,
            limit_area="inside"
        )

        result = pd.DataFrame(
            index=daily_gwl.index
        )

        result.index.name = "date"

        result["gwl_m"] = daily_gwl


        # ====================================================
        # SMOOTH GROUNDWATER LEVEL
        # ====================================================

        result["gwl_smoothed_m"] = (
            result["gwl_m"]
            .rolling(
                window=smoothing_days,
                center=True,
                min_periods=1
            )
            .median()
        )


        # ====================================================
        # WATER-TABLE RISE
        # ====================================================

        groundwater_change_m = (
            result["gwl_smoothed_m"]
            .diff()
        )

        if gwl_type.lower() == "head":

            result["water_table_rise_m"] = (
                groundwater_change_m
            )

        elif gwl_type.lower() == "depth":

            result["water_table_rise_m"] = (
                -groundwater_change_m
            )

        else:

            raise ValueError(
                "gwl_type must be 'head' or 'depth'."
            )


        # Keep only positive rises above threshold
        result["water_table_rise_m"] = (
            result["water_table_rise_m"]
            .where(
                result["water_table_rise_m"]
                >= minimum_rise_m,
                0
            )
        )


        # ====================================================
        # DAILY WTF RECHARGE
        # ====================================================

        result["recharge_mm_per_day"] = (
            sy
            * result["water_table_rise_m"]
            * 1000
        )

        result["station_id"] = station_id
        result["Sy"] = sy

        result = result.reset_index()


        # ====================================================
        # MONTHLY RECHARGE IN MM/MONTH
        # ====================================================

        monthly = (
            result
            .set_index("date")
            .resample("MS")
            .agg(

                monthly_recharge_mm_per_month=(
                    "recharge_mm_per_day",
                    lambda values:
                    values.sum(min_count=1)
                ),

                valid_gwl_days=(
                    "gwl_m",
                    "count"
                ),

                recharge_event_days=(
                    "water_table_rise_m",
                    lambda values:
                    (values > 0).sum()
                )
            )
            .reset_index()
        )

        monthly["year"] = (
            monthly["date"].dt.year
        )

        monthly["month"] = (
            monthly["date"].dt.month
        )

        monthly["station_id"] = station_id
        monthly["Sy"] = sy


        # ====================================================
        # ANNUAL RECHARGE IN MM/YEAR
        # ====================================================

        annual = (
            result
            .set_index("date")
            .resample("YS")
            .agg(

                annual_recharge_mm_per_year=(
                    "recharge_mm_per_day",
                    lambda values:
                    values.sum(min_count=1)
                ),

                valid_gwl_days=(
                    "gwl_m",
                    "count"
                ),

                recharge_event_days=(
                    "water_table_rise_m",
                    lambda values:
                    (values > 0).sum()
                ),

                mean_groundwater_level_m=(
                    "gwl_m",
                    "mean"
                ),

                minimum_groundwater_level_m=(
                    "gwl_m",
                    "min"
                ),

                maximum_groundwater_level_m=(
                    "gwl_m",
                    "max"
                )
            )
            .reset_index()
        )

        annual["year"] = (
            annual["date"].dt.year
        )

        annual["station_id"] = station_id
        annual["Sy"] = sy


        # ====================================================
        # ANNUAL DATA COVERAGE
        # ====================================================

        annual["days_in_year"] = (
            annual["year"]
            .apply(
                lambda year:
                366
                if pd.Timestamp(
                    year=int(year),
                    month=12,
                    day=31
                ).is_leap_year
                else 365
            )
        )

        annual["coverage_percent"] = (
            annual["valid_gwl_days"]
            / annual["days_in_year"]
            * 100
        )

        annual["valid_for_summary"] = (
            annual["coverage_percent"]
            >= minimum_annual_coverage
        )

        valid_annual = annual[
            annual["valid_for_summary"]
        ].copy()


        # ====================================================
        # SAVE INDIVIDUAL STATION RESULTS
        # ====================================================

        result.to_csv(
            daily_folder
            / f"{station_id}_daily_recharge_mm_per_day.csv",
            index=False
        )

        monthly.to_csv(
            monthly_folder
            / f"{station_id}_monthly_recharge_mm_per_month.csv",
            index=False
        )

        annual.to_csv(
            annual_folder
            / f"{station_id}_annual_recharge_mm_per_year.csv",
            index=False
        )


        # ====================================================
        # PLOT 1: GROUNDWATER LEVEL
        # ====================================================

        plt.figure(
            figsize=(12, 5)
        )

        plt.plot(
            result["date"],
            result["gwl_m"],
            linewidth=0.7,
            label="Observed groundwater level"
        )

        plt.plot(
            result["date"],
            result["gwl_smoothed_m"],
            linewidth=1,
            label="Smoothed groundwater level"
        )

        plt.title(
            f"Groundwater-Level Time Series – Station {station_id}"
        )

        plt.xlabel("Date")
        plt.ylabel("Groundwater level (m)")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            plot_folder
            / f"{station_id}_01_groundwater_level.png",
            dpi=plot_dpi,
            bbox_inches="tight"
        )

        plt.close()


        # ====================================================
        # PLOT 2: DAILY RECHARGE
        # ====================================================

        plt.figure(
            figsize=(12, 5)
        )

        plt.plot(
            result["date"],
            result["recharge_mm_per_day"],
            linewidth=0.7
        )

        plt.title(
            f"Daily WTF Recharge – Station {station_id}"
        )

        plt.xlabel("Date")
        plt.ylabel("Recharge (mm/day)")
        plt.grid(alpha=0.3)
        plt.tight_layout()

        plt.savefig(
            plot_folder
            / f"{station_id}_02_daily_recharge.png",
            dpi=plot_dpi,
            bbox_inches="tight"
        )

        plt.close()


        # ====================================================
        # PLOT 3: ANNUAL RECHARGE
        # ====================================================

        plt.figure(
            figsize=(10, 5)
        )

        plt.bar(
            annual["year"].astype(str),
            annual["annual_recharge_mm_per_year"]
        )

        mean_annual_recharge = (
            valid_annual[
                "annual_recharge_mm_per_year"
            ]
            .mean()
        )

        if pd.notna(mean_annual_recharge):

            plt.axhline(
                mean_annual_recharge,
                linestyle="--",
                linewidth=1.2,
                label=(
                    f"Mean = "
                    f"{mean_annual_recharge:.1f} mm/year"
                )
            )

        plt.title(
            f"Annual WTF Recharge – Station {station_id}"
        )

        plt.xlabel("Year")
        plt.ylabel("Annual recharge (mm/year)")
        plt.xticks(rotation=45)
        plt.grid(axis="y", alpha=0.3)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            plot_folder
            / f"{station_id}_03_annual_recharge_mm_per_year.png",
            dpi=plot_dpi,
            bbox_inches="tight"
        )

        plt.close()


        # ====================================================
        # PLOT 4: MONTHLY CLIMATOLOGY
        # ====================================================

        monthly_climatology = (
            monthly
            .groupby("month")[
                "monthly_recharge_mm_per_month"
            ]
            .mean()
            .reindex(range(1, 13))
        )

        plt.figure(
            figsize=(9, 5)
        )

        plt.bar(
            monthly_climatology.index,
            monthly_climatology.values
        )

        plt.title(
            f"Mean Monthly Recharge – Station {station_id}"
        )

        plt.xlabel("Month")
        plt.ylabel("Mean recharge (mm/month)")
        plt.xticks(range(1, 13))
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()

        plt.savefig(
            plot_folder
            / f"{station_id}_04_monthly_climatology.png",
            dpi=plot_dpi,
            bbox_inches="tight"
        )

        plt.close()


        # ====================================================
        # STATION-WISE SUMMARY
        # ====================================================

        annual_values = (
            valid_annual[
                "annual_recharge_mm_per_year"
            ]
            .dropna()
        )

        summary_records.append({

            "station_id":
                station_id,

            "Sy":
                sy,

            "first_date":
                result["date"].min(),

            "last_date":
                result["date"].max(),

            "valid_years":
                len(annual_values),

            "mean_annual_recharge_mm_per_year":
                annual_values.mean(),

            "median_annual_recharge_mm_per_year":
                annual_values.median(),

            "minimum_annual_recharge_mm_per_year":
                annual_values.min(),

            "maximum_annual_recharge_mm_per_year":
                annual_values.max(),

            "standard_deviation_mm_per_year":
                annual_values.std(),

            "mean_coverage_percent":
                annual["coverage_percent"].mean()
        })


        # ====================================================
        # STORE COMBINED RESULTS
        # ====================================================

        all_daily.append(result)
        all_monthly.append(monthly)
        all_annual.append(annual)

        processing_log.append({

            "file":
                file_path.name,

            "station_id":
                station_id,

            "status":
                "Processed",

            "message":
                (
                    f"Date column={date_column}; "
                    f"GWL column={gwl_column}; "
                    f"Sy={sy}; "
                    f"valid years={len(valid_annual)}"
                )
        })

        print(
            f"Processed station: {station_id}"
        )


    except Exception as error:

        processing_log.append({

            "file":
                file_path.name,

            "station_id":
                station_id,

            "status":
                "Error",

            "message":
                str(error)
        })

        print(
            f"Error for {file_path.name}: {error}"
        )


# ============================================================
# 12. CREATE PROCESSING LOG
# ============================================================

log_df = pd.DataFrame(
    processing_log
)

log_df.to_csv(
    log_folder
    / "processing_log.csv",
    index=False
)

if len(all_daily) == 0:

    raise ValueError(
        "No station was processed. "
        "Check the processing log."
    )


# ============================================================
# 13. COMBINE ALL-STATION RESULTS
# ============================================================

all_daily_df = pd.concat(
    all_daily,
    ignore_index=True
)

all_monthly_df = pd.concat(
    all_monthly,
    ignore_index=True
)

all_annual_df = pd.concat(
    all_annual,
    ignore_index=True
)

summary_df = pd.DataFrame(
    summary_records
)


# ============================================================
# 14. SAVE COMBINED CSV FILES
# ============================================================

all_daily_df.to_csv(
    output_folder
    / "all_stations_daily_recharge_mm_per_day.csv",
    index=False
)

all_monthly_df.to_csv(
    output_folder
    / "all_stations_monthly_recharge_mm_per_month.csv",
    index=False
)

all_annual_df.to_csv(
    output_folder
    / "all_stations_annual_recharge_mm_per_year.csv",
    index=False
)

summary_df.to_csv(
    output_folder
    / "station_wise_mean_annual_recharge_mm_per_year.csv",
    index=False
)

station_lookup.to_csv(
    output_folder
    / "station_specific_yield_lookup.csv",
    index=False
)


# ============================================================
# 15. STATION-WISE MEAN ANNUAL RECHARGE PLOT
# ============================================================

plot_summary = summary_df.dropna(
    subset=[
        "mean_annual_recharge_mm_per_year"
    ]
).copy()

plot_summary = plot_summary.sort_values(
    "mean_annual_recharge_mm_per_year",
    ascending=False
)

plt.figure(
    figsize=(
        max(
            12,
            len(plot_summary) * 0.4
        ),
        6
    )
)

plt.bar(
    plot_summary["station_id"].astype(str),
    plot_summary[
        "mean_annual_recharge_mm_per_year"
    ]
)

plt.title(
    "Station-Wise Mean Annual WTF Recharge"
)

plt.xlabel(
    "Groundwater station ID"
)

plt.ylabel(
    "Mean annual recharge (mm/year)"
)

plt.xticks(
    rotation=90
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    plot_folder
    / "all_stations_mean_annual_recharge_mm_per_year.png",
    dpi=plot_dpi,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 16. ANNUAL RECHARGE HEATMAP
# ============================================================

valid_heatmap_data = all_annual_df[
    all_annual_df["valid_for_summary"]
].copy()

heatmap = valid_heatmap_data.pivot_table(
    index="station_id",
    columns="year",
    values="annual_recharge_mm_per_year",
    aggfunc="mean"
)

if not heatmap.empty:

    plt.figure(
        figsize=(
            max(
                10,
                len(heatmap.columns) * 0.7
            ),
            max(
                6,
                len(heatmap.index) * 0.3
            )
        )
    )

    image = plt.imshow(
        heatmap.values,
        aspect="auto"
    )

    plt.colorbar(
        image,
        label="Annual recharge (mm/year)"
    )

    plt.xticks(
        range(len(heatmap.columns)),
        heatmap.columns,
        rotation=45
    )

    plt.yticks(
        range(len(heatmap.index)),
        heatmap.index
    )

    plt.title(
        "Annual WTF Recharge Heatmap"
    )

    plt.xlabel("Year")
    plt.ylabel("Groundwater station ID")
    plt.tight_layout()

    plt.savefig(
        plot_folder
        / "annual_recharge_heatmap_mm_per_year.png",
        dpi=plot_dpi,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 17. REGIONAL ANNUAL RECHARGE TIME SERIES
# ============================================================

regional_annual = (
    valid_heatmap_data
    .groupby(
        "year",
        as_index=False
    )
    .agg(

        mean_recharge_mm_per_year=(
            "annual_recharge_mm_per_year",
            "mean"
        ),

        median_recharge_mm_per_year=(
            "annual_recharge_mm_per_year",
            "median"
        ),

        minimum_recharge_mm_per_year=(
            "annual_recharge_mm_per_year",
            "min"
        ),

        maximum_recharge_mm_per_year=(
            "annual_recharge_mm_per_year",
            "max"
        ),

        stations_available=(
            "station_id",
            "nunique"
        )
    )
)

regional_annual.to_csv(
    output_folder
    / "regional_annual_recharge_mm_per_year.csv",
    index=False
)

if not regional_annual.empty:

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        regional_annual["year"],
        regional_annual[
            "mean_recharge_mm_per_year"
        ],
        marker="o",
        label="Mean"
    )

    plt.plot(
        regional_annual["year"],
        regional_annual[
            "median_recharge_mm_per_year"
        ],
        marker="s",
        label="Median"
    )

    plt.title(
        "Regional Annual WTF Recharge"
    )

    plt.xlabel("Year")
    plt.ylabel("Recharge (mm/year)")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        plot_folder
        / "regional_annual_recharge_mm_per_year.png",
        dpi=plot_dpi,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 18. CREATE EXCEL WORKBOOK
# ============================================================

excel_file = (
    output_folder
    / "WTF_Recharge_Results.xlsx"
)

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    station_lookup.to_excel(
        writer,
        sheet_name="Station_Sy",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Station_Summary",
        index=False
    )

    all_annual_df.to_excel(
        writer,
        sheet_name="Annual_mm_year",
        index=False
    )

    all_monthly_df.to_excel(
        writer,
        sheet_name="Monthly_mm_month",
        index=False
    )

    regional_annual.to_excel(
        writer,
        sheet_name="Regional_Annual",
        index=False
    )

    log_df.to_excel(
        writer,
        sheet_name="Processing_Log",
        index=False
    )


# ============================================================
# 19. SAVE METHOD INFORMATION
# ============================================================

method_text = f"""
WATER TABLE FLUCTUATION RECHARGE ANALYSIS

Daily recharge:
Recharge_mm_per_day =
Sy × positive daily water-table rise in metres × 1000

Monthly recharge:
Monthly recharge is the sum of daily recharge within each month.

Annual recharge:
Annual recharge is the sum of daily recharge within each calendar year.

Analysis period:
{start_date} to {end_date}

Groundwater-level type:
{gwl_type}

Groundwater-level unit:
{gwl_unit}

Minimum accepted groundwater rise:
{minimum_rise_m} m

Maximum interpolated gap:
{maximum_gap_days} days

Smoothing window:
{smoothing_days} days

Minimum annual coverage:
{minimum_annual_coverage} percent
""".strip()

with open(
    output_folder
    / "README_Method_and_Settings.txt",
    mode="w",
    encoding="utf-8"
) as text_file:

    text_file.write(
        method_text
    )


# ============================================================
# 20. CREATE VERIFIED ZIP FILE
# ============================================================

if zip_file.exists():
    zip_file.unlink()

created_zip = shutil.make_archive(
    base_name=str(
        zip_file.with_suffix("")
    ),
    format="zip",
    root_dir=str(
        output_folder.parent
    ),
    base_dir=output_folder.name
)

created_zip = Path(
    created_zip
)

if not created_zip.exists():

    raise FileNotFoundError(
        "ZIP file could not be created."
    )


# ============================================================
# 21. FINAL SUMMARY
# ============================================================

processed_count = (
    log_df["status"]
    .eq("Processed")
    .sum()
)

skipped_count = (
    log_df["status"]
    .eq("Skipped")
    .sum()
)

error_count = (
    log_df["status"]
    .eq("Error")
    .sum()
)

print("\n" + "=" * 70)
print("WTF RECHARGE ANALYSIS COMPLETED")
print("=" * 70)

print(
    f"Processed stations: {processed_count}"
)

print(
    f"Skipped files: {skipped_count}"
)

print(
    f"Files with errors: {error_count}"
)

print(
    f"\nOutput folder:\n{output_folder}"
)

print(
    f"\nExcel workbook:\n{excel_file}"
)

print(
    f"\nZIP file:\n{created_zip}"
)

print(
    f"\nZIP size: "
    f"{created_zip.stat().st_size / (1024 ** 2):.2f} MB"
)

print("=" * 70)


# ============================================================
# 22. DISPLAY TABLES
# ============================================================

print(
    "\nStation-wise mean annual recharge in mm/year:"
)

display(
    summary_df.head(20)
)

print(
    "\nAnnual recharge time series in mm/year:"
)

display(
    all_annual_df.head(30)
)

print(
    "\nProcessing log:"
)

display(
    log_df.head(30)
)


# ============================================================
# 23. DISPLAY DOWNLOAD LINKS
# ============================================================

print(
    "\nDownload complete ZIP file:"
)

display(
    FileLink(
        str(created_zip),
        result_html_prefix="Download ZIP: "
    )
)

print(
    "\nDownload Excel workbook:"
)

display(
    FileLink(
        str(excel_file),
        result_html_prefix="Download Excel: "
    )
)


# ============================================================
# 24. SHOW FILES IN KAGGLE WORKING DIRECTORY
# ============================================================

print(
    "\nFiles saved in /kaggle/working:"
)

for item in sorted(
    Path("/kaggle/working").glob("*")
):

    print(item)

Detected columns in Station_ID.csv:
['GW_ID', 'Sy']


,GW_ID,Sy
0,43420072,0.21
1,43435089,0.21
2,44393876,0.14
3,44406436,0.21
4,44406480,0.11



Station ID column used: GW_ID
Specific-yield column used: Sy

Valid station-Sy records: 176


,station_id,Sy
0,43420072,0.21
1,43435089,0.21
2,44393876,0.14
3,44406436,0.21
4,44406480,0.11
5,44416552,0.11
6,44425470,0.21
7,44445035,0.10
8,44527351,0.10
9,44533080,0.11



Groundwater files found: 380

Processing 1/380: 43420072_1980_2022.csv
Processed station: 43420072

Processing 2/380: 43425100_1980_2012.csv

Processing 3/380: 43425103_1980_2022.csv

Processing 4/380: 43435074_1980_2022.csv

Processing 5/380: 43435083_1980_2022.csv

Processing 6/380: 43435085_1980_2022.csv

Processing 7/380: 43435089_1980_2022.csv
Processed station: 43435089

Processing 8/380: 44393876_1990_2022.csv
Processed station: 44393876

Processing 9/380: 44400361_1_1989_2022.csv

Processing 10/380: 44400865_1_1990_2022.csv

Processing 11/380: 44401566_1_1988_2019.csv

Processing 12/380: 44406436_1980_2022.csv
Processed station: 44406436

Processing 13/380: 44406477_1988_2022.csv

Processing 14/380: 44406480_1989_2022.csv
Processed station: 44406480

Processing 15/380: 44410382_1_1988_2022.csv

Processing 16/380: 44411766_1_1988_2022.csv

Processing 17/380: 44416468_1_1988_2022.csv

Processing 18/380: 44416485_1980_2022.csv

Processing 19/380: 44416534_1984_2022.csv

Processin

...


Download complete ZIP file:


/kaggle/working/WTF_Recharge_Complete_Results.zip


Download Excel workbook:


/kaggle/working/WTF_Recharge_Results/WTF_Recharge_Results.xlsx


Files saved in /kaggle/working:
/kaggle/working/.virtual_documents
/kaggle/working/WTF_Recharge_Complete_Results.zip
/kaggle/working/WTF_Recharge_Results
